Lab Problem Statement: Human Activity Recognition (HAR) using WISDM
Problem Definition:
Develop a Deep Learning model to classify human physical activities based on tri-axial accelerometer data. You are provided with the WISDM (Wireless Sensor Data Mining) dataset, which contains accelerometer recordings from 36 users performing 6 distinct activities.
Objective:
Preprocess the raw time-series data using a Sliding Window technique (segmentation).
Build a 1D Convolutional Neural Network (CNN) or LSTM model to classify the activities.
Evaluate the model's accuracy and visualize the Confusion Matrix.
Target Activities: Walking, Jogging, Upstairs, Downstairs, Sitting, Standing.
Colab Implementation
You can copy-paste the code below directly into a Google Colab cell. It includes a command to automatically download the dataset so you don't need to manually upload files.

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout


In [27]:
df=pd.read_csv('/content/time_series_data_human_activities.csv')


In [28]:
df.head()


In [29]:
df.shape


In [30]:
print(df['activity'].value_counts())


In [31]:
le = LabelEncoder()
df['activity_encoded'] = le.fit_transform(df['activity'])

print("Label Mapping:")
for i, label in enumerate(le.classes_):
    print(label, "->", i)


In [32]:
def sliding_window(data, window_size=128, step_size=64):
    X, y = [], []

    for i in range(0, len(data) - window_size, step_size):
        window = data.iloc[i:i+window_size]
        X.append(window[['x-axis', 'y-axis', 'z-axis']].values)
        y.append(window['activity_encoded'].mode()[0])

    return np.array(X), np.array(y)

X, y = sliding_window(df)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1,3)).reshape(X_train.shape)
X_test  = scaler.transform(X_test.reshape(-1,3)).reshape(X_test.shape)


In [34]:
model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(128,3)),
    MaxPooling1D(2),
    Conv1D(128, kernel_size=3, activation='relu'),
    MaxPooling1D(2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [35]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)


In [36]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)


In [37]:
y_pred = np.argmax(model.predict(X_test), axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - HAR (CNN)")
plt.show()
